# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entity references use their unique Croissant `@id`s for reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant

## 1. Data Loading

In this section, we load the dataset metadata and records using the `mlcroissant` library. This step parses the Croissant JSON-LD schema into a Pythonic data model exposing metadata and record set structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.date_published}")
print(f"Citation: {dataset.metadata.cite_as}")
print(f"License: {dataset.metadata.license}")
print(f"Fields: {dataset.metadata.fields_overview}") if hasattr(dataset.metadata, 'fields_overview') else None

## 2. Data Overview

This section reviews the available record sets, all field definitions, and lists their `@id`s. Use these IDs in subsequent data extraction and processing steps.

- *Record sets* are the core tables present in the dataset.
- *Fields* are the columns/features associated with each record set.

Let's enumerate all top-level record sets, show their IDs, and their available fields (columns) by `@id`.

In [ ]:
# List all record sets by @id
print("Available record sets (by @id):")
record_sets = list(dataset.list_record_sets())  # Each is (id, rs_obj)
for rec_id, rec in record_sets:
    print(f"  - {rec_id}: {getattr(rec, 'name', '')}")
    # List all fields for this record set
    fields = list(rec.list_fields())
    print("    Fields (by @id):")
    for field_id, field in fields:
        print(f"      - {field_id}: {getattr(field, 'name', '')} ({getattr(field, 'data_type', '')})")
    print("-")  # Visual separator

## 3. Data Extraction

In this step, we extract data from a record set into a pandas DataFrame for analysis. Use the record set and field `@id`s from above.

Below, we show how to load all record sets into DataFrames, indexed by their record set `@id`. We demonstrate the extraction using the primary record set (e.g., for protein quantification) whose `@id` is likely similar to `'https://api.app.sen.science/frontiers/7154140/<recordset-uuid>'`.

> **Note:** Copy the exact `@id` from the overview output above! For the purpose of this demonstration, we assume an available record set and pick its `@id` variable.


In [ ]:
# Define the list of record set @ids based on overview results
record_set_ids = [rec_id for rec_id, rec in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Each yielded record is a dict with {field_id: value}
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    dataframes[record_set_id] = df
    print(f"Record set {record_set_id} loaded with {len(df)} rows and {df.shape[1]} columns.")

# Select main record set for the next step (replace with an actual id from your data overview)
main_record_set_id = record_set_ids[0]  # Use first one for demonstration
# Show the first few columns and rows
print('Columns in record set:', dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We now perform data processing on extracted records. All columns are referenced by their `@id`. We'll cover filtering, normalization, and grouping.

- **Filtering:** Remove records where a selected numeric field is below a threshold.
- **Normalization:** Scale a numeric column using standard score (z-score).
- **Grouping:** Aggregate records by a categorical field (e.g., sample type or experiment group).

First, list numeric-like fields (`@id`) available in your selected record set, then choose one.

In [ ]:
df = dataframes[main_record_set_id]
# Helper: Print all available columns (likely referenced by @id)
print("All fields (columns) in main record set:")
for col in df.columns:
    print(f"  - {col}")

# Example: Select one numeric column (replace below with actual column @id from previous step)
numeric_field_id = df.select_dtypes(include=['float', 'int']).columns[0] if df.select_dtypes(include=['float', 'int']).shape[1] else df.columns[0]  # fallback
print(f"Using numeric field for EDA: {numeric_field_id}")

# Try a threshold analysis
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0  # or set to a known value
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Add normalized column
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    normcol = f"{numeric_field_id}_normalized"
    filtered_df[normcol] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} (mean 0, stdev 1):")
    print(filtered_df[[numeric_field_id, normcol]].head())
else:
    print(f"{numeric_field_id} is not numeric, skipping normalization.")

# Attempt grouping by a likely categorical field (by @id)
# Try to pick a non-numeric column
group_field_candidates = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"\nGrouped mean by {group_field_id} (only numeric fields):")
    print(grouped_df.head())
else:
    print('No non-numeric field to group by in this record set.')

## 5. Visualization

Visualize key relationships and distributions in the dataset. Use field references via their `@id`s.

For example, plot the distribution of the main numeric field and a comparison between groups (if possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If grouped, plot mean by group field (bar plot)
if group_field_candidates:
    mean_by_group = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    plt.figure(figsize=(10,4))
    sns.barplot(y=mean_by_group.index, x=mean_by_group)
    plt.title(f'Mean of {numeric_field_id} by {group_field_id}')
    plt.xlabel(f'Mean {numeric_field_id}')
    plt.ylabel(group_field_id)
    plt.show()

## 6. Conclusion

- This notebook demonstrated how to use `mlcroissant` to load, explore, and process a Croissant-annotated FAIR dataset by referencing all data components by their canonical `@id` fields.
- We reviewed available record sets, fields, and iteratively analyzed the data with transformations and visualizations.
- For advanced analysis (e.g., differential expression, protein annotation joins), continue using DataFrames and field references as shown.

> **Tip**: Always reference fields and entities by their `@id` in FAIR, Croissant, or MLCommons-style data pipelines for maximum reproducibility and schema clarity!